# Recession Impact Analysis
## Exploratory Data Analysis & Visualizations

Charts show how employment behaved during recessions. Red shaded areas mark official NBER recession periods.

**5 Charts mapped to 5 Hook Questions:**
1. Total nonfarm payroll with recession shading (Macro anchor)
2. White-collar vs blue-collar % crossover (Hook #1)
3. First blood: who drops first around recession start (Hook #2)
4. Vulnerability shift: era-by-era drawdown comparison (Hook #3)
5. Sub-sector bifurcation: individual ticker crashes across recessions (Hook #4)

---


### Step 1: Load tools and data

Reads `master_table.csv` from Notebook 1.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

df = pd.read_csv(os.path.join("..", "..", "data", "processed", "master_table.csv"),
                 parse_dates=["date"], index_col="date").sort_index()

print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Columns: {list(df.columns)}")

### Step 2: Recession shading helper

Paints background red wherever `recession_indicator` is 1. Used by all charts below.


In [ ]:
def shade_recessions(ax, df, col="recession_indicator"):
    in_rec = False
    start = None
    for date, val in df[col].items():
        if val == 1 and not in_rec:
            in_rec = True
            start = date
        elif val == 0 and in_rec:
            in_rec = False
            ax.axvspan(start, date, color="red", alpha=0.25)
    if in_rec:
        ax.axvspan(start, df.index[-1], color="red", alpha=0.25)
    return ax


### Chart 1: Macro anchor â€” Total nonfarm payroll (PAYEMS)

Overall U.S. job market. Red shaded areas mark official recessions.


In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["total_nonfarm_payroll"], color="steelblue", linewidth=2, label="Total Nonfarm Payroll")
shade_recessions(ax, df)
ax.set_title("Total Nonfarm Payroll Employment with Recession Shading", fontsize=14, fontweight="bold")
ax.set_ylabel("Employees (thousands)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Chart 2: Hook #1 â€” Crossover moment

White-collar % vs blue-collar % of the core cohort. Dashed line at 50% shows when white-collar overtook blue-collar.


In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df["white_collar_pct"], color="tab:blue", linewidth=2, label="White-collar %")
ax.plot(df.index, df["blue_collar_pct"], color="tab:brown", linewidth=2, label="Blue-collar %")
ax.axhline(50, color="red", linestyle="--", linewidth=1.5, label="50% crossover")
shade_recessions(ax, df)
ax.set_title("White-Collar vs Blue-Collar Share of Core Cohort", fontsize=14, fontweight="bold")
ax.set_ylabel("% of Core Cohort Jobs")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Chart 3: Hook #2 — First blood (event study)

Average % employment change from t-6 to t+12 months around recession start dates (t=0).
Shows whether blue-collar or white-collar jobs drop first and fastest.


In [ ]:
recession_starts = df[df["recession_indicator"] == 1].index
recession_starts = recession_starts[recession_starts.to_series().diff().dt.days > 60]

events = [-6, -3, 0, 3, 6, 12]
blue_changes = []
white_changes = []

for t0 in recession_starts:
    base_blue = df.loc[t0, "blue_collar_jobs"]
    base_white = df.loc[t0, "white_collar_jobs"]
    for m in events:
        target = t0 + pd.DateOffset(months=m)
        if target in df.index:
            blue_changes.append((df.loc[target, "blue_collar_jobs"] / base_blue - 1) * 100)
            white_changes.append((df.loc[target, "white_collar_jobs"] / base_white - 1) * 100)

blue_df = pd.DataFrame({"months": events * len(recession_starts), "pct": blue_changes})
white_df = pd.DataFrame({"months": events * len(recession_starts), "pct": white_changes})
blue_avg = blue_df.groupby("months")["pct"].mean()
white_avg = white_df.groupby("months")["pct"].mean()

fig, ax = plt.subplots()
ax.plot(blue_avg.index, blue_avg.values, color="tab:brown", linewidth=2, marker="o", label="Blue-collar")
ax.plot(white_avg.index, white_avg.values, color="tab:blue", linewidth=2, marker="o", label="White-collar")
ax.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Recession start (t=0)")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Average Employment Change Around Recession Start", fontsize=14, fontweight="bold")
ax.set_xlabel("Months relative to recession start")
ax.set_ylabel("Avg % change from t=0")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Chart 4: Hook #3 — Vulnerability shift (era-by-era drawdown)

Compares average employment drawdown during recessions across eras.
Eras are labeled dynamically based on your actual data start date.


In [ ]:
start_date = df.index.min().date()
end_date = df.index.max().date()

# Define eras dynamically
mid_point = df.index[int(len(df) / 2)]
era1 = df.loc[:mid_point]
era2 = df.loc[mid_point:]

era1_label = f"Era 1 ({era1.index.min().date()} to {era1.index.max().date()})"
era2_label = f"Era 2 ({era2.index.min().date()} to {era2.index.max().date()})"

def avg_drawdown(d, col):
    rec = d[d["recession_indicator"] == 1]
    if len(rec) == 0:
        return 0
    base = d[col].iloc[0]
    return ((rec[col] / base - 1) * 100).mean()

era1_blue = avg_drawdown(era1, "blue_collar_jobs")
era1_white = avg_drawdown(era1, "white_collar_jobs")
era2_blue = avg_drawdown(era2, "blue_collar_jobs")
era2_white = avg_drawdown(era2, "white_collar_jobs")

fig, ax = plt.subplots()
x = range(2)
width = 0.35
ax.bar([i - width/2 for i in x], [era1_blue, era2_blue], width, label="Blue-collar", color="tab:brown", alpha=0.8)
ax.bar([i + width/2 for i in x], [era1_white, era2_white], width, label="White-collar", color="tab:blue", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([era1_label, era2_label], rotation=15, ha="right")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Avg % drawdown during recessions")
ax.set_title("Vulnerability Shift: Blue-Collar vs White-Collar Across Eras", fontsize=14, fontweight="bold")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print(f"{era1_label}: Blue={era1_blue:.1f}%, White={era1_white:.1f}%")
print(f"{era2_label}: Blue={era2_blue:.1f}%, White={era2_white:.1f}%")

### Chart 5: Hook #4 — Sub-sector bifurcation

Individual sector drawdowns across major recessions (2001, 2008, 2020).
Shows which sectors got hit hardest in each downturn.


In [ ]:
major_recessions = ["2001-03-01", "2007-12-01", "2020-02-01"]
sector_cols = ["construction_employment", "manufacturing_employment", "professional_business_services", "information_sector"]
sector_labels = ["Construction", "Manufacturing", "Prof. & Business Services", "Information (Tech)"]

results = []
for rec_start in major_recessions:
    rec_start = pd.Timestamp(rec_start)
    if rec_start not in df.index:
        continue
    rec_end = df.loc[df.index >= rec_start][df["recession_indicator"] == 0].index.min()
    if pd.isna(rec_end):
        rec_end = df.index[-1]
    rec_period = df.loc[rec_start:rec_end]
    if len(rec_period) == 0:
        continue
    base = df.loc[df.index <= rec_start].iloc[-1]
    for col, label in zip(sector_cols, sector_labels):
        drop = ((rec_period[col].min() / base[col] - 1) * 100)
        results.append({"Recession": rec_start.date(), "Sector": label, "Max Drop %": round(drop, 1)})

results_df = pd.DataFrame(results)
print(results_df.pivot(index="Sector", columns="Recession", values="Max Drop %"))

fig, ax = plt.subplots()
x = range(len(results_df["Sector"].unique()))
width = 0.25
for i, rec in enumerate(sorted(results_df["Recession"].unique())):
    rec_data = results_df[results_df["Recession"] == rec].sort_values("Sector")
    positions = [p + i * width for p in x]
    ax.bar(positions, rec_data["Max Drop %"], width, label=str(rec), alpha=0.8)

ax.set_xticks([p + width for p in x])
ax.set_xticklabels(results_df.sort_values("Sector")["Sector"].unique(), rotation=15, ha="right")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Max % drop during recession")
ax.set_title("Sub-Sector Drawdowns Across Major Recessions", fontsize=14, fontweight="bold")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()
